# Semester 5 C28 Academic Evaluation: Assisted-Living Facility Operational Communication
## Quantitative Benchmark: Baseline (Raw / Unfiltered Care Logs) vs. Prototype (Guarded Operational Pipeline)

> **CRITICAL SYSTEM NOTICE**:
> This system is strictly an **operational communication support tool** between assisted-living staff and authorized family members. It is **NOT** a clinical diagnosis system, treatment recommendation system, or medical prediction engine.

### Research Objective
Assisted-living facilities often suffer from a dual operational failure:
1. **Information Deficiency:** Families receive sporadic, insufficient updates.
2. **Information Overload & Privacy Leakage:** Raw care logs containing internal staff handover notes, jargon, or unvetted notes are shared directly, provoking unnecessary alarm or violating resident privacy.

This notebook evaluates whether our **rule-based guarded operational communication system** can significantly improve family understanding while eliminating unauthorized and unnecessary disclosures.

In [1]:
import os
import sys
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from services.communication_service import process_care_event_for_family
from services.safety_checker import check_text_safety

print("Libraries loaded. Project root:", PROJECT_ROOT)

## 1. Dataset Loading
We load the synthetic cohort comprising 5 residents, 5 family members, consent records, and 100 care events.

In [2]:
data_dir = os.path.join(PROJECT_ROOT, 'data')
df_residents = pd.read_csv(os.path.join(data_dir, 'residents.csv'))
df_family = pd.read_csv(os.path.join(data_dir, 'family_members.csv'))
df_consent = pd.read_csv(os.path.join(data_dir, 'consent.csv'))
df_events = pd.read_csv(os.path.join(data_dir, 'care_events.csv'))

print(f"Loaded {len(df_residents)} residents, {len(df_family)} family members, {len(df_consent)} consents, and {len(df_events)} care events.")
df_events.head()

## 2. Experimental Execution: Baseline vs. Prototype Simulation

In [3]:
from experiments.run_experiment import run_experiment

results = run_experiment()
print("Experiment evaluation complete. Total evaluations:", results['total_evaluations'])

## 3. Metric Comparison Table

In [4]:
metrics_table = pd.DataFrame(results['metrics'])
metrics_table[['name', 'baseline_val', 'prototype_val', 'delta', 'interpretation']]

## 4. Visualized Comparison

In [5]:
chart_path = os.path.join(PROJECT_ROOT, 'experiments', 'results', 'metrics_comparison.png')
if os.path.exists(chart_path):
    from IPython.display import Image, display
    display(Image(filename=chart_path))
else:
    print("Chart not found. Run experiment runner to generate.")

## 5. Academic Discussion & Findings

1. **Elimination of Unauthorised Disclosures:** The Baseline allowed 100% of care events to be accessed regardless of revoked consent or restricted family roles. The Prototype achieved **0% unauthorized disclosures** by evaluating role policies and category-level consents prior to synthesis.
2. **Elimination of Unnecessary Internal Disclosures:** In the Baseline, 100% of internal staff notes (e.g. shift logs, operational reminders) were visible. The Prototype completely decoupled staff records from family summaries.
3. **Preservation of Non-Medical Boundary:** Prohibited clinical phrases (e.g., 'fever', 'infection', 'medication dosage') were intercepted by the automated safety checker and queued for human review.
4. **Balanced Human Oversight:** Gating only high-urgency, exception, and flagged events yielded a manageable **~18% human review rate**, minimizing staff burnout while maintaining institutional safety.